# 00A · 智能驾驶系统全景：你到底在开发什么模型？

本项目假设你已经具备 Python 和深度学习基础。本 notebook 不讲反向传播或如何调用 PyTorch，而是先建立自动驾驶领域的共同语言：`ego`、`actor`、`scene`、`ODD`、sensor、perception、tracking、prediction、planning、control 和 safety envelope。

如果没有这张地图，后面的 BEV、tracking 或 Transformer 很容易变成孤立的模型名。本节的目标是：看到一个岗位描述或一段模型代码时，知道它位于哪条系统接口上、输入输出是什么、错误会如何传到下游。

**完成后你应该能回答：**

- ego vehicle、other actor、scene、scenario 有什么区别？
- perception 输出为什么不能直接等同于 planning 输入？
- open-loop、closed-loop 和 safety monitor 分别观察什么？
- 为什么 L4 讨论必须同时出现 ODD、fallback 和 scenario regression？


## 1. 用接口而不是模型名理解系统

| 层 | 典型输入 | 典型输出 | 失败会影响什么 |
|---|---|---|---|
| Sensor / calibration | 原始图像、点云、雷达、定位、时间戳 | 对齐后的 sensor bundle | 坐标错、时间错、数据缺失 |
| Perception | sensor bundle | object、lane、occupancy、map element | 漏检、误检、位置/速度不准 |
| Tracking / state estimation | 多帧 observation、ego-motion | 带 ID 的 agent state | ID switch、延迟、漂移 |
| Prediction | agent state、map、traffic context | 多模态 future trajectories | 未来分布漏掉关键模式 |
| Planning | scene state、route、constraints | ego trajectory / maneuver | 碰撞、越界、不可执行 |
| Control | planned trajectory、vehicle state | steering、acceleration、brake | 跟踪误差、舒适性、稳定性 |
| Safety / runtime | health、confidence、TTC、latency | continue、degrade、minimal risk | 需要 fallback 或 ODD exit |

这些层不一定对应独立的神经网络；现代系统可能把多个层放进一个 learned model，但接口、指标和失效分析仍然存在。


In [ ]:
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

@dataclass
class Module:
    name: str
    input_contract: str
    output_contract: str
    evidence: str

modules = [
    Module("perception", "sensor bundle", "objects / occupancy", "detection + geometry"),
    Module("tracking", "multi-frame objects", "agent state + ID", "association + temporal error"),
    Module("prediction", "agent state + map", "future trajectories", "ADE/FDE + miss rate"),
    Module("planning", "scene + route + constraints", "ego trajectory", "collision + progress"),
    Module("control", "trajectory + vehicle state", "actuation", "tracking + comfort"),
    Module("safety", "health + uncertainty + TTC", "continue / degrade", "recall + response latency"),
]
interface_table = pd.DataFrame([m.__dict__ for m in modules])
display(interface_table)


In [ ]:
# 一个最小 scene：ego 在原点，其他 actor 在 ego 坐标系中表达。
ego = np.array([0.0, 0.0])
actors = pd.DataFrame([
    {"actor_id": "lead", "x_m": 18.0, "y_m": 0.2, "speed_mps": 7.5, "kind": "vehicle"},
    {"actor_id": "cut_in", "x_m": 12.0, "y_m": 2.8, "speed_mps": 5.0, "kind": "vehicle"},
    {"actor_id": "pedestrian", "x_m": 9.0, "y_m": -3.2, "speed_mps": 1.4, "kind": "vulnerable"},
])

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(*ego, s=120, marker="^", label="ego")
for _, actor in actors.iterrows():
    ax.scatter(actor.x_m, actor.y_m, s=80, label=f"{actor.actor_id} ({actor.kind})")
    ax.annotate(actor.actor_id, (actor.x_m, actor.y_m), xytext=(5, 5), textcoords="offset points")
ax.axhline(0, color="gray", linewidth=0.8)
ax.set(xlabel="ego-forward x / m", ylabel="ego-left y / m", title="A scene is more than an image")
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.15))
plt.tight_layout()


In [ ]:
from ipywidgets import FloatSlider, IntSlider, interact

def inspect_scene(sensor_dropout=0.0, perception_noise=0.2, planner_delay_ms=120):
    visible = np.random.default_rng(4).random(len(actors)) > sensor_dropout
    measured = actors.loc[visible].copy()
    measured["x_m"] += np.random.default_rng(5).normal(0, perception_noise, len(measured))
    measured["y_m"] += np.random.default_rng(6).normal(0, perception_noise, len(measured))
    end_to_end_ms = 80 + planner_delay_ms + 60
    print(f"visible actors: {len(measured)}/{len(actors)}")
    print(f"toy sensor→perception→planning→control budget: {end_to_end_ms} ms")
    print("system question:", "degrade / minimal risk" if sensor_dropout > 0.35 or end_to_end_ms > 250 else "continue")

interact(
    inspect_scene,
    sensor_dropout=FloatSlider(min=0, max=0.8, step=0.05, value=0.0, description="dropout"),
    perception_noise=FloatSlider(min=0, max=1.5, step=0.1, value=0.2, description="noise / m"),
    planner_delay_ms=IntSlider(min=20, max=300, step=10, value=120, description="planner ms"),
)


## 2. 领域检查点

1. 如果一个模型输出 BEV occupancy，它属于 perception 还是 planning？说明你的接口判断。
2. 如果同一个行人连续三帧被检测到但 ID 不一致，为什么这不是单纯的 detection accuracy 问题？
3. 把 `sensor_dropout` 改到高值，应该由哪个模块决定是否继续运行？为什么不能只看模型 softmax confidence？

**下一步**：`00b` 进入传感器、坐标系和时间契约；然后再进入 `01` 的 SE(3)/标定/投影实现。
